[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/04_PDF%E6%A6%82%E7%8E%87%E5%AF%86%E5%BA%A6%E5%88%86%E4%BD%88.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：密度曲線高到超過 1，是不是機率超過 100%？想問「落在 −1 到 1 之間」該怎麼算？**

- **用在流程哪裡**：連續數值（氣溫、身高、預測誤差）的區間機率計算。
- **讀完要能做到**：能用面積與 CDF 算連續區間機率，不把密度高度當成單點機率。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[數據結構](../名詞解釋/02_數據結構/README.md)、[評估指標](../名詞解釋/09_評估指標/README.md)

# 04｜PDF 與 CDF：機率藏在面積裡

連續隨機變數在區間內有無限多個值，所以**單一精確值的機率是 0**（例如身高恰為 170.000…公分）。機率只存在於區間裡，用面積表示。

**一、PDF：密度不是機率**

機率密度函數 $f(x)$ 有兩條鐵律：$f(x) \ge 0$，且**整條曲線下面積等於 1**（$\int f(x)dx = 1$）。重點有三：

- **高度可以超過 1**：密度單位是「原單位的倒數」。標準差縮到 0.2 時曲線又窄又高，峰值約 2，本篇程式會親手算給你看——但小區間機率 $f \times 寬度$ 仍在 0～1 之間。
- **直方圖要對齊才能比**：`density=True` 的直方圖總**面積**約為 1（不是柱高總和為 1），才能和 PDF 疊在同一尺度比較。樣本再多也不保證每根柱都貼著理論線。
- **有限繪圖範圍只是截取**：圖只畫 −4～4，常態的支持範圍是整條實數線，沒有「到此為止」。

**二、CDF：把面積變成可計算的差**

累積分布函數 $F(x) = P(X \le x)$ 是「從左邊一路累積到 x 的面積」。區間機率就是兩端相減：

$$P(a \le X \le b) = F(b) - F(a) = \int_a^b f(x)dx$$

標準常態 $P(-1 \le X \le 1) \approx 68.27\%$，本篇左圖塗綠色面積、右圖用 `cdf(1)-cdf(-1)` 算出同一個數字，兩種看法互相驗證。

**三、生活比喻**：PDF 像地形等高線的陡峭程度，陡不代表面積大；CDF 像存錢筒累積金額，問「這段存了多少」就是兩個時間點相減。

回歸模型只輸出一個預測值（如 25°C）時，不能憑空宣稱「23～27°C 有幾成」——要談區間機率，必須另建並檢查預測分布或區間方法。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib import font_manager
from IPython.display import display

# 本機優先使用教材字型；Colab 只下載這一個檔案。
candidates = [Path('ChineseFont.ttf'), Path('../ChineseFont.ttf'),
              Path('機率統計/ChineseFont.ttf')]
font_path = next((p for p in candidates if p.is_file()), None)
if font_path is None:
    font_path = Path('ChineseFont.ttf')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/ChineseFont.ttf',
        font_path)
font_manager.fontManager.addfont(str(font_path))
font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
sns.set_theme(style='whitegrid', font=font_name, rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：', font_name)

## 01｜讓理論密度與抽樣直方圖對齊

**為什麼做**：以標準常態模擬資料，直方圖用 density=True 才能和 PDF 比較。

**怎麼看**：直方圖的總面積約為 1，不是柱高總和為 1；資料更多也不是每根柱必然貼近理論。

In [ ]:
samples = np.random.default_rng(42).normal(0, 1, 2000)
x = np.linspace(-4, 4, 600)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples, bins=35, density=True, color=GREEN, alpha=.25, label='抽樣密度')
ax.plot(x, stats.norm.pdf(x), color=GREEN, lw=2, label='理論 PDF')
ax.set(title='同一尺度才能比較：密度直方圖與 PDF', xlabel='數值', ylabel='機率密度')
ax.legend()
plt.show()

## 02｜區間機率＝曲線下的面積

**為什麼做**：用 CDF 的兩個端點相減，計算 −1 到 1 的機率。

**怎麼看**：CDF(x)=P(X≤x)。左圖綠色面積約 68.27%；右圖用 CDF(1)−CDF(−1) 算出同一件事。連續單點機率為 0。

In [ ]:
a, b = -1, 1
interval_probability = stats.norm.cdf(b) - stats.norm.cdf(a)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x, stats.norm.pdf(x), color=GREEN)
mask = (x >= a) & (x <= b)
axes[0].fill_between(x[mask], stats.norm.pdf(x[mask]), color=GREEN, alpha=.25)
axes[0].set(title=f'區間面積：{interval_probability:.2%}', xlabel='數值', ylabel='機率密度')
axes[1].plot(x, stats.norm.cdf(x), color=PURPLE)
axes[1].scatter([a, b], stats.norm.cdf([a, b]), color=ORANGE, zorder=3)
axes[1].set(title='CDF：累積到這裡的機率', xlabel='數值', ylabel='累積機率', ylim=(0, 1.05))
plt.show()
print(f'P(−1 ≤ X ≤ 1) = {interval_probability:.4f}')
assert np.isclose(interval_probability, .682689492, atol=1e-8)

## 03｜高度可以大於 1，面積仍是 1

**為什麼做**：將標準差改小，觀察變窄、變高的 PDF。密度的單位是原數值單位的倒數。

**怎麼看**：橘線峰值接近 1.995，大於 1；但是同一小區間的機率仍介於 0～1。有限繪圖範圍只是截取，常態支持範圍沒有到此停止。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for sigma, color in [(1, GREEN), (.2, ORANGE)]:
    ax.plot(x, stats.norm.pdf(x, scale=sigma), color=color,
            label=f'標準差 {sigma:g}')
ax.set(title='密度變高，不代表機率超過 1', xlabel='數值', ylabel='機率密度')
ax.legend()
plt.show()
peak = stats.norm.pdf(0, scale=.2)
small_interval = stats.norm.cdf(.1, scale=.2) - stats.norm.cdf(-.1, scale=.2)
print(f'中心密度：{peak:.3f}；−0.1 到 0.1 的機率：{small_interval:.2%}')
assert peak > 1 and 0 <= small_interval <= 1

## 接回機器學習

回歸模型只輸出一個預測值時，不能憑空宣稱區間機率。要說「有幾成落在某範圍」，需另建立並檢查預測分布或區間的方法。

## 換你解讀

1. PDF 高度與 CDF 高度各代表什麼？
2. density=True 的直方圖，哪一個總和應接近 1？
3. 模型只預測 25°C，就能說 23～27°C 的機率嗎？

### 請 AI 幫你做一個小實驗

請 AI 把陰影區間從 −1～1 改成 −2～2，先估計機率增加還是降低，再用 CDF 差驗證。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)